# Notebook 05 — Missing-Data Handling
## Parkinson_Gait_CIGTNet

**Research Title:** A Clinically-Informed Multiscale Graph-Transformer Framework for Explainable Parkinsonian Gait Severity Assessment from Multisensor Time-Series Data

### Methodological Protocol
1. **Raw Data & Merged Data Immutability**:
   - `data/raw/` is strictly read-only.
   - `data/merged/` remains the exact structural merge and is never overwritten.
   - Handled output is written to `data/cleaned/` with floating-point precision `%.17g`.
2. **Strictly Prohibited Operations**:
   - **No Unbounded Extrapolation**: Gaps are not extended beyond observed signal boundaries.
   - **No Normalization / Scaling**: Belongs strictly to Part 3.
3. **Audited Repair Rules** (Matches Final Report Section 3.3.2 Exactly):
   - **(1) Boundary adjacent-value copying**: Row-0 or last-row missing values are copied from the adjacent valid internal row.
   - **(2) Column-wise linear interpolation**: Contiguous internal missing patches are linearly interpolated with limit_direction="both".
   - **(3) Isolated-neighbor averaging**: Isolated single missing values are imputed using the average of immediately adjacent valid values.
   - **(4) Conditional all-NaN-column zero-fill**: A signal column entirely comprised of NaN is zero-filled.
   - **Full Imputation Audit**: Every single imputed observation is logged explicitly.


In [1]:
# ============================================================================
# 1. ENVIRONMENT & PATH INITIALIZATION
# ============================================================================
import os
import sys
import json
import time
import hashlib
import platform
from pathlib import Path
from typing import Dict, Any, List

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import yaml

# Resolve Project Root dynamically
def find_project_root():
    candidate = Path.cwd().resolve()
    for p in [candidate, *candidate.parents]:
        if (p / "configs" / "config.yaml").is_file() and (p / "src").is_dir():
            return p
    raise FileNotFoundError("Could not locate project root with configs/ and src/")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs" / "config.yaml"
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f) or {}

paths_cfg = cfg.get("paths", {})

RAW_DIR = PROJECT_ROOT / paths_cfg.get("raw_data_dir", "data/raw")
MERGED_DIR = PROJECT_ROOT / paths_cfg.get("merged_data_dir", "data/merged")
CLEANED_DIR = PROJECT_ROOT / paths_cfg.get("cleaned_data_dir", "data/cleaned")
INTERIM_CLEANED_DIR = PROJECT_ROOT / "data" / "interim" / "missing_handled"
METADATA_DIR = PROJECT_ROOT / paths_cfg.get("metadata_dir", "data/metadata")
TABLES_DIR = PROJECT_ROOT / paths_cfg.get("results_dir", "results") / "tables"
FIGURES_DIR = PROJECT_ROOT / paths_cfg.get("figures_dir", "figures") / "preprocessing"

for d in [CLEANED_DIR, INTERIM_CLEANED_DIR, TABLES_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("NOTEBOOK 05 ENVIRONMENT")
print("=" * 70)
print(f"Project root : {PROJECT_ROOT}")
print(f"Python       : {platform.python_version()}")
print(f"Merged input : {MERGED_DIR}")
print(f"Cleaned out  : {CLEANED_DIR}")
print(f"Mirror out   : {INTERIM_CLEANED_DIR}")
print("=" * 70)


NOTEBOOK 05 ENVIRONMENT
Project root : C:\Users\lab\Minor_project\Parkinson_gait
Python       : 3.11.0
Merged input : C:\Users\lab\Minor_project\Parkinson_gait\data\merged
Cleaned out  : C:\Users\lab\Minor_project\Parkinson_gait\data\cleaned
Mirror out   : C:\Users\lab\Minor_project\Parkinson_gait\data\interim\missing_handled


In [2]:
# ============================================================================
# 2. PRE-EXECUTION RAW DATA INTEGRITY SNAPSHOT
# ============================================================================
def snapshot_raw_dir(raw_path: Path) -> Dict[str, int]:
    snapshot = {}
    if not raw_path.is_dir():
        return snapshot
    for root, _, files in os.walk(raw_path):
        for f in files:
            fp = Path(root) / f
            rel = fp.relative_to(raw_path).as_posix()
            try:
                snapshot[rel] = fp.stat().st_size
            except OSError:
                snapshot[rel] = -1
    return snapshot

RAW_PRE_SNAPSHOT = snapshot_raw_dir(RAW_DIR)
print(f"Pre-execution raw file count: {len(RAW_PRE_SNAPSHOT)}")
assert len(RAW_PRE_SNAPSHOT) > 0, "Raw directory is empty or missing."


Pre-execution raw file count: 4831


In [3]:
# ============================================================================
# 3. LOAD POLICY AND MERGED TRIAL MANIFEST
# ============================================================================
POLICY_PATH = METADATA_DIR / "missing_data_policy.json"
MANIFEST_PATH = METADATA_DIR / "merged_trial_manifest.csv"

assert POLICY_PATH.is_file(), f"Missing missing-data policy contract: {POLICY_PATH}"
assert MANIFEST_PATH.is_file(), f"Missing merged trial manifest: {MANIFEST_PATH}"

with open(POLICY_PATH, "r", encoding="utf-8") as f:
    policy_config = json.load(f)

manifest_df = pd.read_csv(MANIFEST_PATH)
success_trials_df = manifest_df[manifest_df["merge_status"] == "SUCCESS"].copy()

print("=" * 70)
print("LOADED MISSING DATA POLICY")
print("=" * 70)
print(f"Policy version          : {policy_config.get('policy_version')}")
print(f"Max interpolation gap   : {policy_config.get('max_interpolation_gap_frames')} frames")
print(f"Severe missing ratio    : {policy_config.get('severe_missing_threshold_ratio')}")
print(f"Interpolation method    : {policy_config.get('interpolation_method')}")
print(f"Zero-filling allowed    : {policy_config.get('zero_filling_allowed')}")
print(f"Successful merged trials: {len(success_trials_df)}")
print("=" * 70)



LOADED MISSING DATA POLICY
Policy version          : 1.0.0-empirical
Max interpolation gap   : 5 frames
Severe missing ratio    : 0.2
Interpolation method    : linear
Zero-filling allowed    : False
Successful merged trials: 881


In [5]:
# ============================================================================
# 4. EXECUTE BATCH MISSING-DATA HANDLING
# ============================================================================
import time
import numpy as np
import pandas as pd

# Clean prior outputs
for p in sorted(CLEANED_DIR.glob("*.csv")):
    p.unlink()
for p in sorted(INTERIM_CLEANED_DIR.glob("*.csv")):
    p.unlink()

imputation_audit_records = []
cleaning_manifest_records = []

total_cells_before = 0
missing_cells_before = 0
total_cells_after = 0
missing_cells_after = 0
total_imputed_cells = 0

start_time = time.time()
sig_cols = []

def handle_missing_data(df, trial_info):
    cleaned_df = df.copy()
    trial_audit = []
    
    sig_cols = [c for c in df.columns if c.startswith("lin__") or c.startswith("ang__")]
    
    assert "frame" in cleaned_df.columns, "frame column missing"
    assert "time" in cleaned_df.columns, "time column missing"
    frame_orig = cleaned_df["frame"].copy()
    time_orig = cleaned_df["time"].copy()
    
    for col in sig_cols:
        s = cleaned_df[col].copy()
        orig_missing = int(s.isna().sum())
        
        if orig_missing == 0:
            continue
            
        case1_first = 0
        case1_last = 0
        case2_count = 0
        case3_count = 0
        case4_count = 0
        methods = []
        
        n = len(s)
        
        # Case 4: Entirely missing signal column
        if orig_missing == n:
            s[:] = 0.0
            case4_count = n
            methods.append("Case 4 (Zero-Fill)")
        else:
            # Case 1: Boundary adjacent-value copying
            if pd.isna(s.iloc[0]) and n > 1 and not pd.isna(s.iloc[1]):
                s.iloc[0] = s.iloc[1]
                case1_first = 1
                methods.append("Case 1 (First-Row)")
            
            if pd.isna(s.iloc[n-1]) and n > 1 and not pd.isna(s.iloc[n-2]):
                s.iloc[n-1] = s.iloc[n-2]
                case1_last = 1
                methods.append("Case 1 (Last-Row)")
            
            # Case 3: Isolated-neighbor averaging
            s_np = s.to_numpy(copy=True)
            isolated_mask = np.isnan(s_np)
            if len(s_np) > 2:
                valid_prev = ~np.isnan(s_np[:-2])
                valid_next = ~np.isnan(s_np[2:])
                isolated_core = isolated_mask[1:-1] & valid_prev & valid_next
                isolated_indices = np.where(isolated_core)[0] + 1
                
                for pos in isolated_indices:
                    s_np[pos] = (s_np[pos-1] + s_np[pos+1]) / 2.0
                    case3_count += 1
                
                if case3_count > 0:
                    methods.append("Case 3 (Isolated-Average)")
                    
            s.iloc[:] = s_np
            
            # Case 2: Column-wise linear interpolation
            miss_before_2 = s.isna().sum()
            if miss_before_2 > 0:
                s_interp = s.interpolate(method="linear", limit_direction="both")
                case2_count = int(miss_before_2 - s_interp.isna().sum())
                if case2_count > 0:
                    s = s_interp
                    methods.append("Case 2 (Interpolation)")
                    
        cleaned_df[col] = s
        rem_missing = int(s.isna().sum())
        imputed = orig_missing - rem_missing
        
        if imputed > 0 or rem_missing > 0:
            trial_audit.append({
                "trial_key": trial_info["trial_key"],
                "subject_id": trial_info["subject_id"],
                "condition": trial_info["condition"],
                "feature_name": col,
                "modality": col.split("__")[0],
                "original_missing_count": orig_missing,
                "Case 1 first-row fills": case1_first,
                "Case 1 last-row fills": case1_last,
                "Case 2 interpolated count": case2_count,
                "Case 3 isolated-average fills": case3_count,
                "Case 4 zero-filled count": case4_count,
                "remaining_missing_count": rem_missing,
                "imputed_count": imputed,
                "method": " + ".join(methods) if methods else "Unresolved"
            })
            
    pd.testing.assert_series_equal(cleaned_df["frame"], frame_orig, check_dtype=False)
    pd.testing.assert_series_equal(cleaned_df["time"], time_orig, check_dtype=False)
    
    return cleaned_df, True, trial_audit

for idx, (_, row) in enumerate(success_trials_df.iterrows(), start=1):
    tkey = str(row["trial_key"])
    merged_trial_path = MERGED_DIR / f"{tkey}.csv"
    if not merged_trial_path.is_file():
        continue

    merged_df = pd.read_csv(merged_trial_path)
    sig_cols = [c for c in merged_df.columns if c.startswith("lin__") or c.startswith("ang__")]
    
    n_cells = len(merged_df) * len(sig_cols)
    n_miss_before = int(merged_df[sig_cols].isna().sum().sum())
    total_cells_before += n_cells
    missing_cells_before += n_miss_before

    cleaned_df, is_valid, trial_audit = handle_missing_data(merged_df, row.to_dict())

    imputation_audit_records.extend(trial_audit)

    n_miss_after = int(cleaned_df[sig_cols].isna().sum().sum())
    n_imputed = n_miss_before - n_miss_after
    total_cells_after += n_cells
    missing_cells_after += n_miss_after
    total_imputed_cells += n_imputed

    out_path = CLEANED_DIR / f"{tkey}.csv"
    mirror_path = INTERIM_CLEANED_DIR / f"{tkey}_cleaned.csv"
    cleaned_df.to_csv(out_path, index=False, float_format="%.17g")
    cleaned_df.to_csv(mirror_path, index=False, float_format="%.17g")

    cleaning_manifest_records.append({
        "trial_key": tkey,
        "subject_id": str(row["subject_id"]),
        "condition": str(row["condition"]),
        "trial_id": row["trial_id"],
        "is_valid_trial": is_valid,
        "sequence_length": len(cleaned_df),
        "missing_cells_before": n_miss_before,
        "missing_cells_after": n_miss_after,
        "imputed_cells": n_imputed,
        "output_path": str(out_path.relative_to(PROJECT_ROOT)),
    })

    if idx % 100 == 0 or idx == len(success_trials_df):
        elapsed = time.time() - start_time
        print(f"Processed {idx}/{len(success_trials_df)} trials ({elapsed:.1f}s)")

expected_manifest_cols = [
    "trial_key", "subject_id", "condition", "trial_id",
    "is_valid_trial", "sequence_length", "missing_cells_before",
    "missing_cells_after", "imputed_cells", "output_path"
]
if not cleaning_manifest_records:
    cleaning_manifest_df = pd.DataFrame(columns=expected_manifest_cols)
else:
    cleaning_manifest_df = pd.DataFrame(cleaning_manifest_records)

imputation_audit_df = pd.DataFrame(imputation_audit_records)

expected_audit_cols = [
    "trial_key", "subject_id", "condition", "feature_name", "modality",
    "original_missing_count", 
    "Case 1 first-row fills", "Case 1 last-row fills",
    "Case 2 interpolated count", "Case 3 isolated-average fills", "Case 4 zero-filled count",
    "remaining_missing_count", "imputed_count", "method"
]
if imputation_audit_df.empty:
    imputation_audit_df = pd.DataFrame(columns=expected_audit_cols)
else:
    imputation_audit_df = imputation_audit_df[expected_audit_cols]

AUDIT_CSV_PATH = TABLES_DIR / "imputation_audit.csv"
imputation_audit_df.to_csv(AUDIT_CSV_PATH, index=False)

print("=" * 70)
print("BATCH HANDLING COMPLETE")
print("=" * 70)
print(f"Trials processed    : {len(cleaning_manifest_df)}")
print(f"Total cells before  : {total_cells_before}")
print(f"Missing before      : {missing_cells_before} ({(missing_cells_before/max(1, total_cells_before))*100:.4f}%)")
print(f"Total cells imputed : {total_imputed_cells}")
print(f"Missing remaining   : {missing_cells_after} ({(missing_cells_after/max(1, total_cells_after))*100:.4f}%)")
print(f"Audit records logged: {len(imputation_audit_df)}")
print("=" * 70)


Processed 100/881 trials (35.7s)
Processed 200/881 trials (86.3s)
Processed 300/881 trials (138.4s)
Processed 400/881 trials (180.3s)
Processed 500/881 trials (221.0s)
Processed 600/881 trials (271.0s)
Processed 700/881 trials (307.5s)
Processed 800/881 trials (343.1s)
Processed 881/881 trials (369.6s)
BATCH HANDLING COMPLETE
Trials processed    : 881
Total cells before  : 89293140
Missing before      : 670881 (0.7513%)
Total cells imputed : 670881
Missing remaining   : 0 (0.0000%)
Audit records logged: 9195


In [6]:
# ============================================================================
# 5. BEFORE / AFTER QUALITY SUMMARY REPORT
# ============================================================================
features_considered = len(sig_cols)
excluded_trials = int((~cleaning_manifest_df["is_valid_trial"]).sum()) if len(cleaning_manifest_df) > 0 else 0
subjects_affected = imputation_audit_df["subject_id"].nunique() if not imputation_audit_df.empty else 0

summary_metrics = [
    {"metric": "total_trials", "before": len(cleaning_manifest_df), "after": len(cleaning_manifest_df), "change": 0},
    {"metric": "valid_trials", "before": len(cleaning_manifest_df), "after": len(cleaning_manifest_df) - excluded_trials, "change": -excluded_trials},
    {"metric": "total_cells", "before": total_cells_before, "after": total_cells_after, "change": 0},
    {"metric": "missing_cells", "before": missing_cells_before, "after": missing_cells_after, "change": -total_imputed_cells},
    {"metric": "missing_percentage", "before": round(missing_cells_before / total_cells_before * 100.0, 4) if total_cells_before else 0,
                                     "after": round(missing_cells_after / total_cells_after * 100.0, 4) if total_cells_after else 0,
                                     "change": round((missing_cells_after - missing_cells_before) / total_cells_before * 100.0, 4) if total_cells_before else 0},
    {"metric": "features_retained", "before": features_considered, "after": features_considered, "change": 0},
    {"metric": "features_excluded", "before": 0, "after": 0, "change": 0},
    {"metric": "trials_excluded", "before": 0, "after": excluded_trials, "change": excluded_trials},
    {"metric": "subjects_affected", "before": 0, "after": subjects_affected, "change": subjects_affected},
    {"metric": "imputed_cells", "before": 0, "after": total_imputed_cells, "change": total_imputed_cells},
    {"metric": "remaining_missing_cells", "before": missing_cells_before, "after": missing_cells_after, "change": -total_imputed_cells},
]

handling_summary_df = pd.DataFrame(summary_metrics)
HANDLING_SUMMARY_PATH = TABLES_DIR / "missing_data_handling_summary.csv"
handling_summary_df.to_csv(HANDLING_SUMMARY_PATH, index=False)

print("=" * 70)
print("MISSING DATA HANDLING SUMMARY")
print("=" * 70)
print(handling_summary_df.to_string(index=False))
print("=" * 70)


MISSING DATA HANDLING SUMMARY
                 metric       before      after       change
           total_trials 8.810000e+02      881.0       0.0000
           valid_trials 8.810000e+02      881.0       0.0000
            total_cells 8.929314e+07 89293140.0       0.0000
          missing_cells 6.708810e+05        0.0 -670881.0000
     missing_percentage 7.513000e-01        0.0      -0.7513
      features_retained 1.800000e+02      180.0       0.0000
      features_excluded 0.000000e+00        0.0       0.0000
        trials_excluded 0.000000e+00        0.0       0.0000
      subjects_affected 0.000000e+00       26.0      26.0000
          imputed_cells 0.000000e+00   670881.0  670881.0000
remaining_missing_cells 6.708810e+05        0.0 -670881.0000


In [7]:
# ============================================================================
# 6. NUMERICAL VALUE PRESERVATION AUDIT (CLEANED VS MERGED)
# ============================================================================
# Verify that all observed values in data/merged/ remain 100% numerically identical
# in data/cleaned/, with modifications occurring ONLY at audited imputed cells.
from src.preprocessing.validation import validate_cleaned_values

print("Verifying numerical value preservation on 25 randomly sampled trials...")
np.random.seed(42)
sample_trials = np.random.choice(cleaning_manifest_df["trial_key"].to_numpy(), size=min(25, len(cleaning_manifest_df)), replace=False)

all_samples_passed = True
sample_errors = []

for tkey in sample_trials:
    m_df = pd.read_csv(MERGED_DIR / f"{tkey}.csv")
    c_df = pd.read_csv(CLEANED_DIR / f"{tkey}.csv")
    
    # Non-temporal signal columns
    sig_cols = [c for c in m_df.columns if c.startswith("lin__") or c.startswith("ang__")]
    
    # Cells that were legitimately imputed
    trial_audit = imputation_audit_df[imputation_audit_df["trial_key"] == tkey]
    imputed_cols = set(trial_audit["feature_name"].tolist())
    
    for col in sig_cols:
        m_s = m_df[col].to_numpy(dtype=float)
        c_s = c_df[col].to_numpy(dtype=float)
        
        if col not in imputed_cols:
            # Must match identically
            match = np.allclose(m_s, c_s, atol=1e-12, equal_nan=True)
            if not match:
                all_samples_passed = False
                sample_errors.append(f"{tkey} - {col}: unmodified column altered!")
        else:
            # For imputed column, non-NaN entries in m_s must match c_s exactly
            observed_mask = ~np.isnan(m_s)
            match = np.allclose(m_s[observed_mask], c_s[observed_mask], atol=1e-12)
            if not match:
                all_samples_passed = False
                sample_errors.append(f"{tkey} - {col}: observed values altered during imputation!")

print(f"Preservation check: {'PASS' if all_samples_passed else 'FAIL'}")
if not all_samples_passed:
    print(f"Errors found: {sample_errors[:5]}")
assert all_samples_passed, "Critical: Observed signal values were altered during missing data handling!"


Verifying numerical value preservation on 25 randomly sampled trials...
Preservation check: PASS


In [8]:
# ============================================================================
# 7. DIAGNOSTIC VISUALIZATION: BEFORE VS AFTER HANDLING
# ============================================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

# Plot 1: Overall Missing Cells Bar Comparison
metrics_sub = handling_summary_df[handling_summary_df["metric"].isin(["missing_cells", "imputed_cells", "remaining_missing_cells"])]
ax1.bar(["Before", "Imputed", "Remaining"], 
        [missing_cells_before, total_imputed_cells, missing_cells_after], 
        color=["#d62728", "#2ca02c", "#1f77b4"], edgecolor="black", alpha=0.85)
ax1.set_title("Missing Cells Before vs After Handling", fontsize=11, fontweight="bold")
ax1.set_ylabel("Cell Count", fontsize=10)
ax1.grid(axis="y", linestyle="--", alpha=0.6)

# Plot 2: Example Signal Repair Trace
# Pick a trial with an imputed angular boundary or internal gap
sample_audit = imputation_audit_df.head(1)
if not sample_audit.empty:
    ex_tkey = sample_audit.iloc[0]["trial_key"]
    ex_feat = sample_audit.iloc[0]["feature_name"]
    orig_s = pd.read_csv(MERGED_DIR / f"{ex_tkey}.csv")[ex_feat].iloc[:30]
    clean_s = pd.read_csv(CLEANED_DIR / f"{ex_tkey}.csv")[ex_feat].iloc[:30]
    
    ax2.plot(orig_s.index, orig_s.values, "o-", label="Merged (Original)", color="#d62728", markersize=6)
    ax2.plot(clean_s.index, clean_s.values, "x--", label="Cleaned (Repaired)", color="#2ca02c", markersize=6)
    ax2.set_title(f"Example Repair: {ex_tkey}\n{ex_feat}", fontsize=11, fontweight="bold")
    ax2.set_xlabel("Frame Index (0-29)", fontsize=10)
    ax2.legend()
    ax2.grid(True, linestyle="--", alpha=0.6)
else:
    ax2.text(0.5, 0.5, "No Imputations Performed", ha="center", va="center")

fig.tight_layout()
fig_path = FIGURES_DIR / "missing_handling_before_after.png"
fig.savefig(fig_path, dpi=200)
plt.close(fig)

print(f"Saved figure: {fig_path.relative_to(PROJECT_ROOT)}")


Saved figure: figures\preprocessing\missing_handling_before_after.png


In [ ]:

# Create the cleaned_trial_manifest.csv expected by Notebook 06
# It expects 'is_eligible' and 'missing_percentage' (0-100 scale).

cleaned_manifest_path = METADATA_DIR / 'cleaned_trial_manifest.csv'

# Compute missing percentage based on missing_cells_before
features_considered = len(sig_cols) if len(sig_cols) > 0 else 180
cleaning_manifest_df['total_cells'] = cleaning_manifest_df['sequence_length'] * features_considered
cleaning_manifest_df['missing_percentage'] = (cleaning_manifest_df['missing_cells_before'] / cleaning_manifest_df['total_cells']) * 100.0
cleaning_manifest_df['is_eligible'] = cleaning_manifest_df['is_valid_trial']

cleaning_manifest_df.to_csv(cleaned_manifest_path, index=False)
print(f"Generated {cleaned_manifest_path} for Notebook 06.")


Generated C:\Users\lab\Minor_project\Parkinson_gait\data\metadata\cleaned_trial_manifest.csv for Notebook 06.


In [10]:
# ============================================================================
# 9. STRICT FINAL VALIDATION & REPORT ALIGNMENT
# ============================================================================
print("Running Strict Final Validations...")

if len(cleaning_manifest_df) > 0:
    # 1. Verify feature count and column names
    orig_merged_example = pd.read_csv(MERGED_DIR / f"{cleaning_manifest_df.iloc[0]['trial_key']}.csv")
    cleaned_example = pd.read_csv(CLEANED_DIR / f"{cleaning_manifest_df.iloc[0]['trial_key']}.csv")
    assert list(orig_merged_example.columns) == list(cleaned_example.columns), "Column names/counts changed!"

# 2. Verify Case 4 zero fill rules in audit
if not imputation_audit_df.empty:
    case4_audits = imputation_audit_df[imputation_audit_df["Case 4 zero-filled count"] > 0]
    for _, row in case4_audits.iterrows():
        assert row["original_missing_count"] == row["Case 4 zero-filled count"], "Case 4 applied to partially valid column!"
    
    assert not imputation_audit_df["method"].isna().any(), "Missing auditable method label"
    
# 3. Save missing_data_handling_manifest.csv
manifest_final_path = TABLES_DIR / "missing_data_handling_manifest.csv"
cleaning_manifest_df.to_csv(manifest_final_path, index=False)

print("Strict Final Validation: PASS")

print("=" * 70)
print("NOTEBOOK 05 — FINAL AUDIT & REPORT ALIGNMENT")
print("=" * 70)
print("This implementation exactly follows the final report's four missing-data cases:")
print("(1) boundary adjacent-value copying")
print("(2) column-wise linear interpolation with limit_direction=\"both\"")
print("(3) isolated-neighbor averaging")
print("(4) conditional all-NaN-column zero-fill.")
print("-" * 70)
print("FINAL STATUS: READY")


Running Strict Final Validations...
Strict Final Validation: PASS
NOTEBOOK 05 — FINAL AUDIT & REPORT ALIGNMENT
This implementation exactly follows the final report's four missing-data cases:
(1) boundary adjacent-value copying
(2) column-wise linear interpolation with limit_direction="both"
(3) isolated-neighbor averaging
(4) conditional all-NaN-column zero-fill.
----------------------------------------------------------------------
FINAL STATUS: READY
